In [1]:
from peft import PeftModel
from transformers import T5Tokenizer, T5EncoderModel
from clalign.alignment import ProteinSeq, AlignmentResult, align_core
from clalign.plm import PLM
from clalign.metrics import f1score, hec_acc, hec_sov

/home/yrh/CLAlign/.conda/envs/CLAlign-Release/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cd ..

/home/yrh/CLAlign


In [3]:
tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_uniref50')
model = T5EncoderModel.from_pretrained('Rostlab/prot_t5_xl_uniref50').cuda()

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:00<00:00, 40816.42it/s]
[transformers] T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
model = PeftModel.from_pretrained(model, 'src/clalign/CLAlign-ProtT5')
plm = PLM(tokenizer, model, 512)

In [5]:
def align(seq1, seq2):
    embs = plm.get_embs([seq1.seq, seq2.seq])
    return align_core(seq1, seq2, embs[0] @ embs[1].T)

In [6]:
import numpy as np
from pathlib import Path

def test(data):
    name_list, f_list, tms, hacc_list, hsov_list = [], [], [], [], []
    Path(f'results/{data}/clalign-prott5').mkdir(parents=True, exist_ok=True)
    with open(f'data/{data}_hec.csv') as fp, open(f'results/{data}/clalign-prott5.txt', 'w') as fout:
        fp.readline()
        for line in fp:
            name, f1, f2, s1, s2, hec1, hec2, aln1, aln2 = line.strip().split(',')
            name_list.append(name)
            manual = AlignmentResult(seq1 := ProteinSeq(s1), seq2 := ProteinSeq(s2), aln1, aln2)
            aln_res = align(seq1, seq2)
            f_list.append(f_ := f1score(manual, aln_res))
            if len(seq1) == len(hec1) and len(seq2) == len(hec2):
                 hacc_list.append(hacc_ := hec_acc(aln_res, hec1, hec2))
                 hsov_list.append(hsov_ := hec_sov(aln_res, hec1, hec2))
            else:
                 print('hec not match')
                 hacc_ = hsov_ = 0.0
            with open(out_:=(f'results/{data}/clalign-prott5/{name}.txt'), 'w') as faln:
                print('>p1', file=faln)
                print(aln_res.aln1, file=faln)
                print('>p2', file=faln)
                print(aln_res.aln2, file=faln)
            out = !TMalign data/{data}/{name}/{f1} data/{data}/{name}/{f2} -I {out_} -a T
            tms.append(s_:=float(out[17][10:17]))
            print(f'{name}: P: {f_[0]:.3f}, R: {f_[1]:.3f}, F: {f_[2]:.3f}, S: {s_:.5f}, HEC ACC: {hacc_:.5f}, HEC SOV: {hsov_:.5f}')
            print(name, f_[0], f_[1], f_[2], s_, hacc_, hsov_, file=fout, sep='\t')
        p_, r_, f_ = np.asarray(f_list).mean(axis=0)
        print(f'total: {len(f_list)}, P: {p_:.3f}, R: {r_:.3f}, F: {f_:.3f}, TM-score: {np.mean(tms):.5f}, HEC ACC: {np.mean(hacc_list):.5f}, HEC SOV: {np.mean(hsov_list):.5f}')
        return tms

In [7]:
malidup_tms = test('malidup')

d19hca_: P: 0.637, R: 0.644, F: 0.640, S: 0.45028, HEC ACC: 0.17761, HEC SOV: 0.44672
d1a4pa_: P: 0.875, R: 0.875, F: 0.875, S: 0.52345, HEC ACC: 0.56522, HEC SOV: 0.78333
d1a4sa_: P: 0.514, R: 0.514, F: 0.514, S: 0.44569, HEC ACC: 0.37299, HEC SOV: 0.64780
d1a6da1: P: 0.358, R: 0.382, F: 0.370, S: 0.27248, HEC ACC: 0.44898, HEC SOV: 0.50049
d1a8l_1: P: 0.755, R: 0.755, F: 0.755, S: 0.70439, HEC ACC: 0.50442, HEC SOV: 0.77758
d1af2a1: P: 0.674, R: 0.684, F: 0.679, S: 0.53833, HEC ACC: 0.38776, HEC SOV: 0.61971
d1afwb1: P: 0.547, R: 0.557, F: 0.552, S: 0.40383, HEC ACC: 0.31043, HEC SOV: 0.48922
d1ahja_: P: 0.857, R: 0.857, F: 0.857, S: 0.33028, HEC ACC: 0.15152, HEC SOV: 0.41875
d1ahua1: P: 0.676, R: 0.676, F: 0.676, S: 0.38523, HEC ACC: 0.40418, HEC SOV: 0.52024
d1ai3__: P: 0.250, R: 0.325, F: 0.283, S: 0.32083, HEC ACC: 0.35749, HEC SOV: 0.46756
d1aj8a_: P: 0.130, R: 0.162, F: 0.145, S: 0.16373, HEC ACC: 0.38814, HEC SOV: 0.51816
d1ako__: P: 0.794, R: 0.794, F: 0.794, S: 0.46433, HEC

In [8]:
malisam_tms = test('malisam')

d1a05a_d1dgsa3: P: 0.063, R: 0.052, F: 0.057, S: 0.13765, HEC ACC: 0.28409, HEC SOV: 0.47538
d1a05a_d1j71a_: P: 0.000, R: 0.000, F: 0.000, S: 0.14981, HEC ACC: 0.24155, HEC SOV: 0.34150
d1a05a_d1rblm_: P: 0.531, R: 0.544, F: 0.537, S: 0.41913, HEC ACC: 0.45026, HEC SOV: 0.73345
d1a2za_d1ghha_: P: 0.191, R: 0.194, F: 0.193, S: 0.27279, HEC ACC: 0.42045, HEC SOV: 0.58065
d1a2za_d1u9da_: P: 0.247, R: 0.287, F: 0.266, S: 0.29944, HEC ACC: 0.38140, HEC SOV: 0.53558
d1a7j__d1kafa_: P: 0.394, R: 0.500, F: 0.441, S: 0.31969, HEC ACC: 0.35676, HEC SOV: 0.49115
d1a7j__d2if1__: P: 0.579, R: 0.698, F: 0.633, S: 0.34833, HEC ACC: 0.22660, HEC SOV: 0.38557
d1aa7a_d1b68a_: P: 0.000, R: 0.000, F: 0.000, S: 0.15734, HEC ACC: 0.37209, HEC SOV: 0.51931
d1aa7a_d1qkra_: P: 0.151, R: 0.151, F: 0.151, S: 0.14714, HEC ACC: 0.33577, HEC SOV: 0.50426
d1ac5__d1jroa3: P: 0.114, R: 0.111, F: 0.113, S: 0.25762, HEC ACC: 0.27835, HEC SOV: 0.43393
d1ac5__d1vk0a_: P: 0.000, R: 0.000, F: 0.000, S: 0.12670, HEC ACC: 0.1